In [ ]:
import os
import sys
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Ensure we can import from the src directory
sys.path.append(os.path.dirname(os.getcwd()))
from src.discretizer import Discretizer

# Set plotting style
sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = (14, 6)

In [ ]:
# 1. Load the fully processed batch we created earlier
data_path = os.path.join("..", "data", "processed", "static_batch_may2019.csv")
df = pd.read_csv(data_path)

# 2. Enforce Mechanism Bounds (Individual Rationality)
# We only care about jobs with a positive Virtual Value (phi_v > 0)
# Jobs with negative phi_v are mathematically rejected by the Optimization Oracle
valid_df = df[df['phi_v'] > 0].copy()
continuous_phi = valid_df['phi_v'].values

print(f"Total jobs requested: {len(df)}")
print(f"Jobs accepted by Mechanism (phi_v > 0): {len(valid_df)}")
print(f"Maximum Theoretical Revenue (Continuous): ${np.sum(continuous_phi):.2f}")

In [ ]:
K_BINS = 20
discretizer = Discretizer(K_bins=K_BINS)

results = []

# --- 1. Evaluate Uniform Grid ---
start_time = time.perf_counter()
uniform_discrete_phi = discretizer.uniform_grid(continuous_phi)
uniform_time = time.perf_counter() - start_time

uniform_error = discretizer.calculate_discretization_error(continuous_phi, uniform_discrete_phi)
results.append({
    "Method": "Uniform (Arithmetic)",
    "Extracted Revenue ($)": np.sum(uniform_discrete_phi),
    "Revenue Loss ($)": uniform_error,
    "Execution Time (sec)": uniform_time
})

# --- 2. Evaluate Geometric Grid ---
start_time = time.perf_counter()
geometric_discrete_phi = discretizer.geometric_grid(continuous_phi)
geometric_time = time.perf_counter() - start_time

geometric_error = discretizer.calculate_discretization_error(continuous_phi, geometric_discrete_phi)
results.append({
    "Method": "Geometric (Multiplicative)",
    "Extracted Revenue ($)": np.sum(geometric_discrete_phi),
    "Revenue Loss ($)": geometric_error,
    "Execution Time (sec)": geometric_time
})

# Display the performance matrix
results_df = pd.DataFrame(results)
display(results_df)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(18, 6), sharey=True)

# Define common bins for the histograms to make them comparable
plot_bins = np.linspace(0, np.percentile(continuous_phi, 99), 50)

# --- Plot 1: Uniform Grid ---
sns.histplot(continuous_phi, bins=plot_bins, color='lightgray', label='Continuous $\Phi_v$', ax=axes[0], stat='count')
sns.histplot(uniform_discrete_phi, bins=plot_bins, color='blue', alpha=0.6, label='Discrete Assigned Prices', ax=axes[0], stat='count')
axes[0].set_title(f"Uniform Grid (K={K_BINS})\nRevenue Loss: ${uniform_error:.2f}", fontsize=14)
axes[0].set_xlabel("Virtual Value ($)")
axes[0].set_ylabel("Number of Jobs")
axes[0].legend()

# --- Plot 2: Geometric Grid ---
sns.histplot(continuous_phi, bins=plot_bins, color='lightgray', label='Continuous $\Phi_v$', ax=axes[1], stat='count')
sns.histplot(geometric_discrete_phi, bins=plot_bins, color='green', alpha=0.6, label='Discrete Assigned Prices', ax=axes[1], stat='count')
axes[1].set_title(f"Geometric Grid (K={K_BINS})\nRevenue Loss: ${geometric_error:.2f}", fontsize=14)
axes[1].set_xlabel("Virtual Value ($)")
axes[1].legend()

plt.tight_layout()
plt.show()